In [0]:
from pyspark.sql import functions as F

source_path = (
    "/Volumes/clubdata/bronze/landing/football/"
    "matches_2026-08-21.json"
)
bronze_table = "clubdata.bronze.matches_raw"

source_document = (
    spark.read
    .option("multiLine", "true")
    .json(source_path)
)

bronze_matches = (
    source_document
    .select(F.explode("data.matches").alias("payload"))
    .select(
        F.col("payload.match_id")
            .cast("long")
            .alias("source_match_id"),
        F.lit("footballdata.io").alias("source_system"),
        F.lit(source_path).alias("source_file"),
        F.current_timestamp().alias("ingested_at"),
        F.sha2(F.to_json("payload"), 256).alias("record_hash"),
        F.col("payload")
    )
)

display(bronze_matches.limit(5))

In [0]:
(
    bronze_matches.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print(f"Opprettet {bronze_table}")

In [0]:
%sql

SELECT
    COUNT(*) AS raw_rows,
    COUNT(DISTINCT source_match_id) AS distinct_match_ids,
    COUNT_IF(payload IS NULL) AS missing_payloads
FROM clubdata.bronze.matches_raw;

In [0]:
%sql

SELECT
    source_match_id,
    payload.match_date,
    payload.home_team.team_name AS home_team,
    payload.away_team.team_name AS away_team,
    payload.score.home AS home_score,
    payload.score.away AS away_score,
    ingested_at
FROM clubdata.bronze.matches_raw
ORDER BY payload.match_date DESC
LIMIT 10;